# Which rules expanded, returned results, were rejected, or were empty?

This notebook audits **10 treatment queries × 18 rules × 4 hosted environments = 720 rule evaluations**.
It uses the saved main run (`run_002`). The 40 direct lookups are counted separately.

**Expansion and outcome are different dimensions.** An expanded rule can return results,
be rejected, or return an empty callback. We confirm expansion by matching a callback query
graph from the live logs to the release template; the existence of a template file alone is insufficient.

**Returned** means a positive raw callback result count, before ARAGORN filters and merges.
It does not mean a unique chemical answer or a confirmed contribution to the final response.

Running this notebook is offline. No new service queries are submitted.


In [1]:
from pathlib import Path
import json
import hashlib
import sys
from html import escape
import pandas as pd
from IPython.display import display, HTML, Markdown

candidates = [p for base in (Path.cwd(), *Path.cwd().parents)
              for p in (base, base / 'experiments/aragron-exploration/expansion-across-environments',
                        base / 'expansion-across-environments')]
ROOT = next(p for p in candidates if (p / 'scripts/rule_outcomes.py').exists())
RUN = ROOT / 'output/run_002'
analysis = json.loads((RUN / 'rule-outcomes.json').read_text())
assert analysis['source_report_sha256'] == hashlib.sha256((RUN / 'report.json').read_bytes()).hexdigest(), 'Rebuild rule_outcomes.py after updating the source report.'
rows = analysis['records']
LABELS = {'development':'Development', 'ci_staging':'CI / Staging', 'test':'Test', 'production':'Production'}
ENVIRONMENTS = list(LABELS)
COLORS = {'returned':'#dcfce7','rejected':'#fee2e2','empty':'#e5e7eb','incomplete':'#fef3c7','error':'#fef3c7','unknown':'#dbeafe'}
SYMBOLS = {'returned':'R','rejected':'X','empty':'E','incomplete':'I','error':'!','unknown':'?'}

def table(data):
    return pd.DataFrame(data).to_html(index=False, escape=True)

def numbers(values):
    return ', '.join(map(str, values)) if values else 'None'

display(pd.DataFrame([analysis['totals']]))
display(pd.DataFrame([{'Term':k,'Meaning':v} for k,v in analysis['definitions'].items()]))
print('Expanded is a separate total. Returned + rejected + empty + incomplete + error + unknown partitions the rule evaluations.')


Expanded is a separate total. Returned + rejected + empty + incomplete + error + unknown partitions the rule evaluations.


,rule_evaluations,expanded,returned,rejected,empty,incomplete,error,unknown,returned_with_issues
0,720,720,108,320,292,0,0,0,0


,Term,Meaning
0,expanded,Observed callback query graph matched the adve...
1,returned,"Positive raw callback count; pre-merge, not un..."
2,rejected,Logged invalid predicate or unsupported MetaEd...
3,empty,"Exactly one identified callback returned zero,..."
4,incomplete,Zero/unknown result with a timeout or truncati...
5,error,Other logged backend error with no positive ca...
6,unknown,Missing or ambiguous evidence; never silently ...


## Outcome totals for each environment

A rule evaluation is one rule for one target in one environment. Thus 27 returned evaluations
means 27 rule–target combinations had positive callbacks; it is not 27 rules or 27 chemicals.
The reported error count excludes the explicitly classified rejections.


In [2]:
environment_counts = pd.DataFrame(analysis['environment_summary'])
display(environment_counts)
assert all(sum(s[k] for k in ('returned','rejected','empty','incomplete','error','unknown')) == s['rule_evaluations'] for s in analysis['environment_summary'])
agreement = pd.DataFrame(analysis['environment_agreement'])
display(agreement)


,environment,rule_evaluations,expanded,returned,rejected,empty,incomplete,error,unknown,returned_with_issues
0,development,180,180,27,80,73,0,0,0,0
1,ci_staging,180,180,27,80,73,0,0,0,0
2,test,180,180,27,80,73,0,0,0,0
3,production,180,180,27,80,73,0,0,0,0


,query,target_name,all_environments_present,same_expansion_and_outcome_groups
0,q01,pheochromocytoma,True,True
1,q02,"cafe au lait spots, multiple",True,True
2,q03,biotinidase deficiency,True,True
3,q04,Fever,True,True
4,q05,Halitosis,True,True
5,q06,COVID-19,True,True
6,q07,"osteoarthritis, hand",True,True
7,q08,liver carcinoma,True,True
8,q09,mycosis fungoides,True,True
9,q10,Endotoxemia,True,True


## Which rules had each outcome for each target?

The table below combines environments **only if their expanded-rule and outcome lists agree for
that target**. Otherwise each environment is displayed separately. Rejected rules are explicitly
listed; they must not be interpreted as successful zero-result searches.


In [3]:
query_rows = []
for agreement in analysis['environment_agreement']:
    subset = [q for q in analysis['query_summary'] if q['query']==agreement['query']]
    selected = subset[:1] if agreement['same_expansion_and_outcome_groups'] else subset
    for q in selected:
        query_rows.append({'Query':q['query'], 'Target':q['target_name'],
            'Environments':'All four (verified)' if agreement['same_expansion_and_outcome_groups'] else LABELS[q['environment']],
            'Expanded':numbers(q['expanded_rules']), 'Returned':numbers(q['returned_rules']),
            'Rejected':numbers(q['rejected_rules']), 'Empty':numbers(q['empty_rules']),
            'Incomplete / error / unknown':numbers(sorted(q['incomplete_rules']+q['error_rules']+q['unknown_rules']))})
display(pd.DataFrame(query_rows))


,Query,Target,Environments,Expanded,Returned,Rejected,Empty,Incomplete / error / unknown
0,q01,pheochromocytoma,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","6, 10, 15, 18","1, 2, 3, 4, 5, 7, 8, 9","11, 12, 13, 14, 16, 17",None
1,q02,"cafe au lait spots, multiple",All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",10,"1, 2, 3, 4, 5, 7, 8, 9","6, 11, 12, 13, 14, 15, 16, 17, 18",None
2,q03,biotinidase deficiency,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",11,"1, 2, 3, 4, 5, 7, 8, 9","6, 10, 12, 13, 14, 15, 16, 17, 18",None
3,q04,Fever,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","6, 11, 14, 15, 17, 18","1, 2, 3, 4, 5, 7, 8, 9","10, 12, 13, 16",None
4,q05,Halitosis,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","11, 17","1, 2, 3, 4, 5, 7, 8, 9","6, 10, 12, 13, 14, 15, 16, 18",None
5,q06,COVID-19,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","11, 15, 17, 18","1, 2, 3, 4, 5, 7, 8, 9","6, 10, 12, 13, 14, 16",None
6,q07,"osteoarthritis, hand",All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",11,"1, 2, 3, 4, 5, 7, 8, 9","6, 10, 12, 13, 14, 15, 16, 17, 18",None
7,q08,liver carcinoma,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","6, 10, 11, 15, 17, 18","1, 2, 3, 4, 5, 7, 8, 9","12, 13, 14, 16",None
8,q09,mycosis fungoides,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","11, 17","1, 2, 3, 4, 5, 7, 8, 9","6, 10, 12, 13, 14, 15, 16, 18",None
9,q10,Endotoxemia,All four (verified),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",None,"1, 2, 3, 4, 5, 7, 8, 9","6, 10, 11, 12, 13, 14, 15, 16, 17, 18",None


## Status grid: ten targets × eighteen rules, for each environment

**R (count)** = returned raw results; **X** = rejected; **E** = empty callback with no error logged.
**I / ! / ?** indicate incomplete search, other backend error, or unknown evidence.
A `*` on a returned result means additional issues are recorded in the detailed evidence.
Color is supplementary; every cell also has a text label. Hover over a cell for the exact outcome.


In [4]:
for env in ENVIRONMENTS:
    selected = [r for r in rows if r['environment']==env]
    html = '<table><thead><tr><th>Query / target</th>' + ''.join(f'<th>R{n}</th>' for n in range(1,19)) + '</tr></thead><tbody>'
    for query in dict.fromkeys(r['query'] for r in selected):
        query_rows = [r for r in selected if r['query']==query]
        html += '<tr><th style="text-align:left">' + escape(query + ' — ' + query_rows[0]['target_name']) + '</th>'
        by_rule = {r['rule']:r for r in query_rows}
        for number in range(1,19):
            row = by_rule.get(number)
            if row is None:
                html += '<td>?</td>'
                continue
            label = SYMBOLS[row['category']]
            if row['category']=='returned':
                label += ' (' + str(row['raw_result_count']) + ')' + ('*' if row['issue_flags'] else '')
            tooltip = f"Expanded: {row['expanded']}; " + ', '.join(row['outcomes'] + row['issue_flags'])
            html += '<td title="' + escape(tooltip, quote=True) + '" style="padding:6px;white-space:nowrap;color:#111827;background:' + COLORS[row['category']] + '">' + escape(label) + '</td>'
        html += '</tr>'
    html += '</tbody></table>'
    display(HTML('<details open><summary><b>' + LABELS[env] + '</b></summary><div style="overflow-x:auto">' + html + '</div></details>'))


Query / target,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10,R11,R12,R13,R14,R15,R16,R17,R18
q01 — pheochromocytoma,X,X,X,X,X,R (411),X,X,X,R (288),E,E,E,E,R (4),E,E,R (4)
"q02 — cafe au lait spots, multiple",X,X,X,X,X,E,X,X,X,R (119),E,E,E,E,E,E,E,E
q03 — biotinidase deficiency,X,X,X,X,X,E,X,X,X,E,R (17),E,E,E,E,E,E,E
q04 — Fever,X,X,X,X,X,R (9048),X,X,X,E,R (81),E,E,R (23),R (4),E,R (25),R (4)
q05 — Halitosis,X,X,X,X,X,E,X,X,X,E,R (10),E,E,E,E,E,R (51),E
q06 — COVID-19,X,X,X,X,X,E,X,X,X,E,R (153),E,E,E,R (7),E,R (62),R (7)
"q07 — osteoarthritis, hand",X,X,X,X,X,E,X,X,X,E,R (64),E,E,E,E,E,E,E
q08 — liver carcinoma,X,X,X,X,X,R (3357),X,X,X,R (2544),R (104),E,E,E,R (7),E,R (41),R (7)
q09 — mycosis fungoides,X,X,X,X,X,E,X,X,X,E,R (198),E,E,E,E,E,R (3),E
q10 — Endotoxemia,X,X,X,X,X,E,X,X,X,E,E,E,E,E,E,E,E,E


Query / target,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10,R11,R12,R13,R14,R15,R16,R17,R18
q01 — pheochromocytoma,X,X,X,X,X,R (411),X,X,X,R (288),E,E,E,E,R (4),E,E,R (4)
"q02 — cafe au lait spots, multiple",X,X,X,X,X,E,X,X,X,R (119),E,E,E,E,E,E,E,E
q03 — biotinidase deficiency,X,X,X,X,X,E,X,X,X,E,R (17),E,E,E,E,E,E,E
q04 — Fever,X,X,X,X,X,R (9048),X,X,X,E,R (81),E,E,R (23),R (4),E,R (25),R (4)
q05 — Halitosis,X,X,X,X,X,E,X,X,X,E,R (10),E,E,E,E,E,R (51),E
q06 — COVID-19,X,X,X,X,X,E,X,X,X,E,R (153),E,E,E,R (7),E,R (62),R (7)
"q07 — osteoarthritis, hand",X,X,X,X,X,E,X,X,X,E,R (64),E,E,E,E,E,E,E
q08 — liver carcinoma,X,X,X,X,X,R (3357),X,X,X,R (2544),R (104),E,E,E,R (7),E,R (41),R (7)
q09 — mycosis fungoides,X,X,X,X,X,E,X,X,X,E,R (198),E,E,E,E,E,R (3),E
q10 — Endotoxemia,X,X,X,X,X,E,X,X,X,E,E,E,E,E,E,E,E,E


Query / target,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10,R11,R12,R13,R14,R15,R16,R17,R18
q01 — pheochromocytoma,X,X,X,X,X,R (411),X,X,X,R (288),E,E,E,E,R (4),E,E,R (4)
"q02 — cafe au lait spots, multiple",X,X,X,X,X,E,X,X,X,R (119),E,E,E,E,E,E,E,E
q03 — biotinidase deficiency,X,X,X,X,X,E,X,X,X,E,R (17),E,E,E,E,E,E,E
q04 — Fever,X,X,X,X,X,R (9048),X,X,X,E,R (81),E,E,R (23),R (4),E,R (25),R (4)
q05 — Halitosis,X,X,X,X,X,E,X,X,X,E,R (10),E,E,E,E,E,R (51),E
q06 — COVID-19,X,X,X,X,X,E,X,X,X,E,R (153),E,E,E,R (7),E,R (62),R (7)
"q07 — osteoarthritis, hand",X,X,X,X,X,E,X,X,X,E,R (64),E,E,E,E,E,E,E
q08 — liver carcinoma,X,X,X,X,X,R (3357),X,X,X,R (2544),R (104),E,E,E,R (7),E,R (41),R (7)
q09 — mycosis fungoides,X,X,X,X,X,E,X,X,X,E,R (198),E,E,E,E,E,R (3),E
q10 — Endotoxemia,X,X,X,X,X,E,X,X,X,E,E,E,E,E,E,E,E,E


Query / target,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10,R11,R12,R13,R14,R15,R16,R17,R18
q01 — pheochromocytoma,X,X,X,X,X,R (411),X,X,X,R (288),E,E,E,E,R (4),E,E,R (4)
"q02 — cafe au lait spots, multiple",X,X,X,X,X,E,X,X,X,R (119),E,E,E,E,E,E,E,E
q03 — biotinidase deficiency,X,X,X,X,X,E,X,X,X,E,R (17),E,E,E,E,E,E,E
q04 — Fever,X,X,X,X,X,R (9048),X,X,X,E,R (81),E,E,R (23),R (4),E,R (25),R (4)
q05 — Halitosis,X,X,X,X,X,E,X,X,X,E,R (10),E,E,E,E,E,R (51),E
q06 — COVID-19,X,X,X,X,X,E,X,X,X,E,R (153),E,E,E,R (7),E,R (62),R (7)
"q07 — osteoarthritis, hand",X,X,X,X,X,E,X,X,X,E,R (64),E,E,E,E,E,E,E
q08 — liver carcinoma,X,X,X,X,X,R (3357),X,X,X,R (2544),R (104),E,E,E,R (7),E,R (41),R (7)
q09 — mycosis fungoides,X,X,X,X,X,E,X,X,X,E,R (198),E,E,E,E,E,R (3),E
q10 — Endotoxemia,X,X,X,X,X,E,X,X,X,E,E,E,E,E,E,E,E,E


## What happened to each rule across all forty query/environment combinations?

Each rule has forty evaluations: ten targets in four environments.
“Always empty” is restricted to these ten targets. It is not a claim that no matching path exists
anywhere in the graph. A rule returning paths for some targets may be empty for others.


In [5]:
display(pd.DataFrame(analysis['rule_summary']))
print('Always rejected in this sample:', numbers([r['rule'] for r in analysis['rule_summary'] if r['rejected']==r['rule_evaluations']]))
print('Always empty in this sample:', numbers([r['rule'] for r in analysis['rule_summary'] if r['empty']==r['rule_evaluations']]))
print('Returned results for at least one target:', numbers([r['rule'] for r in analysis['rule_summary'] if r['returned']]))


Always rejected in this sample: 1, 2, 3, 4, 5, 7, 8, 9
Always empty in this sample: 12, 13, 16
Returned results for at least one target: 6, 10, 11, 14, 15, 17, 18


,rule,rule_evaluations,expanded,returned,rejected,empty,incomplete,error,unknown,returned_with_issues
0,1,40,40,0,40,0,0,0,0,0
1,2,40,40,0,40,0,0,0,0,0
2,3,40,40,0,40,0,0,0,0,0
3,4,40,40,0,40,0,0,0,0,0
4,5,40,40,0,40,0,0,0,0,0
5,6,40,40,12,0,28,0,0,0,0
6,7,40,40,0,40,0,0,0,0,0
7,8,40,40,0,40,0,0,0,0,0
8,9,40,40,0,40,0,0,0,0,0
9,10,40,40,12,0,28,0,0,0,0


## Why were rules rejected? Which edges did the service flag?

Rule 1 is rejected for an invalid predicate. The other rejected rules have query edges for which
Retriever reports no compatible **MetaEdge**: a supported category/predicate/category pattern.

These messages describe the service's validation or supported schema. They do not independently
establish graph-wide absence. An edge not named by the error has **not** been independently
confirmed present. Full rejection messages and observed graphs are retained for every evaluation.


In [6]:
rejection_rows = []
seen = set()
for row in rows:
    if row['category']!='rejected':
        continue
    for evidence in row['rejections']:
        for edge in evidence['flagged_edges']:
            if 'predicates' not in edge:
                pattern = 'Observed edge details unavailable'
            else:
                pattern = ', '.join(edge['subject_categories']) + ' → ' + ', '.join(edge['predicates']) + ' → ' + ', '.join(edge['object_categories'])
            key = (row['rule'], edge['edge'], pattern, evidence['message'])
            if key in seen:
                continue
            seen.add(key)
            rejection_rows.append({'Rule':row['rule'], 'Flagged edge':edge['edge'], 'Typed edge pattern':pattern,
                                   'Exact service message':evidence['message']})
display(pd.DataFrame(rejection_rows).sort_values(['Rule','Flagged edge']))
print('Development also logs some unknown-constraints warnings. Those warnings are retained and alone do not prove qualifiers were ignored.')


Development also logs some unknown-constraints warnings. Those warnings are retained and alone do not prove qualifiers were ignored.


,Rule,Flagged edge,Typed edge pattern,Exact service message
0,1,edge_0,biolink:Gene → biolink:affects_response_to → b...,Validation Error: Edge `edge_0` has invalid pr...
1,2,edge_2,biolink:ChemicalEntity → biolink:similar_to → ...,MetaEdges could not be found for the following...
2,3,edge_1,biolink:Gene → biolink:genetic_association → b...,MetaEdges could not be found for the following...
3,4,edge_2,biolink:Gene → biolink:genetic_association → b...,MetaEdges could not be found for the following...
4,5,edge_1,biolink:Gene → biolink:genetic_association → b...,MetaEdges could not be found for the following...
5,7,edge_1,biolink:Gene → biolink:genetic_association → b...,MetaEdges could not be found for the following...
6,8,edge_1,biolink:Gene → biolink:genetic_association → b...,MetaEdges could not be found for the following...
7,9,edge_1,biolink:BiologicalProcess → biolink:has_output...,MetaEdges could not be found for the following...
8,9,edge_2,biolink:BiologicalProcess → biolink:has_output...,MetaEdges could not be found for the following...


## Inspect the exact evidence for one query and environment

Change `QUERY` and `ENVIRONMENT` below. Each rule shows its outcome, raw count, observed query
graph, warnings/errors and callback ID. A returned path count is recorded before merge filters;
this view does not attribute final merged answers to individual rules.


In [7]:
QUERY = 'q01'
ENVIRONMENT = 'development'
selected = [r for r in rows if r['query']==QUERY and r['environment']==ENVIRONMENT]
assert selected, 'Choose q01 … q10 and development / ci_staging / test / production.'
display(Markdown('**' + LABELS[ENVIRONMENT] + ' — ' + selected[0]['target_name'] + '**'))
display(pd.DataFrame([{k:r[k] for k in ('rule','expanded','category','raw_result_count','callback_count','issue_flags')} for r in selected]))
for row in selected:
    content = {k:row[k] for k in ('outcomes','callback_ids','rejections','warnings_errors','observed_query_graphs','response_artifact')}
    title = f"Rule {row['rule']}: {row['category']} — raw results: {row['raw_result_count']}"
    display(HTML('<details><summary>' + escape(title) + '</summary><pre>' + escape(json.dumps(content,indent=2)) + '</pre></details>'))


**Development — pheochromocytoma**

,rule,expanded,category,raw_result_count,callback_count,issue_flags
0,1,True,rejected,0,1,[callback includes error logs]
1,2,True,rejected,0,1,[]
2,3,True,rejected,0,1,[]
3,4,True,rejected,0,1,[]
4,5,True,rejected,0,1,[]
5,6,True,returned,411,1,[]
6,7,True,rejected,0,1,[]
7,8,True,rejected,0,1,[]
8,9,True,rejected,0,1,[]
9,10,True,returned,288,1,[]


## Direct lookup is separate

ARAGORN also performs the original question as a direct lookup. Those forty callbacks are excluded
from the 720 rule evaluations above. Direct results explain how a target can receive final answers
even if none of its eighteen inference-rule callbacks returned a result.


In [8]:
direct = pd.DataFrame(analysis['direct_lookups'])
display(direct.pivot(index=['query','target_name'],columns='environment',values='raw_result_count').reindex(columns=ENVIRONMENTS).rename(columns=LABELS))
for limitation in analysis['limitations']:
    print('•', limitation)
print('Detailed export:', (RUN / 'rule-outcomes.csv').resolve())
print('Per-query lists:', (RUN / 'rule-outcomes-by-query.csv').resolve())


• No new HTTP requests; this is a re-analysis of the saved run.
• All counts in this analysis refer to rule-query evaluations or raw callbacks; direct lookups are separate.
• Warnings such as skipping unknown constraints are retained; an empty label does not prove qualifier enforcement.
• Unsupported MetaEdges describe service schema compatibility; they do not establish which individual graph records are absent.
• No final merged answer is attributed to a specific rule from these counts alone.
• A target-specific empty callback does not establish global absence of the rule.
Detailed export: /home/agonzalez/shepherd-myara/experiments/aragron-exploration/expansion-across-environments/output/run_002/rule-outcomes.csv
Per-query lists: /home/agonzalez/shepherd-myara/experiments/aragron-exploration/expansion-across-environments/output/run_002/rule-outcomes-by-query.csv


,environment,Development,CI / Staging,Test,Production
query,target_name,,,,
q01,pheochromocytoma,5,5,5,5
q02,"cafe au lait spots, multiple",0,0,0,0
q03,biotinidase deficiency,1,1,1,1
q04,Fever,92,92,92,92
q05,Halitosis,2,2,2,2
q06,COVID-19,50,50,50,50
q07,"osteoarthritis, hand",4,4,4,4
q08,liver carcinoma,35,35,35,35
q09,mycosis fungoides,23,23,23,23
